In [9]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import csv
import time
import re

PARK_ID = 160
CSV_FILE = "efteling_rides_all_years.csv"

# ------------------------------------
# Helper: build URL from date
# ------------------------------------
def build_url(date_obj):
    return f"https://queue-times.com/parks/{PARK_ID}/calendar/{date_obj.year}/{date_obj.month:02d}/{date_obj.day:02d}"

# ------------------------------------
# Helper: find table under specific H2 title
# ------------------------------------
def extract_table_by_title(soup, title_text):
    h2 = soup.find("h2", string=lambda s: s and title_text.lower() in s.lower())
    if not h2:
        return None
    panel = h2.find_parent("div", class_="panel")
    if not panel:
        return None
    return panel.find("table")

# ------------------------------------
# Helper: convert table rows → dict
# ------------------------------------
def parse_table(table, value_type="float"):
    """
    Returns dict { ride: value }
    where value is cleaned (number only)
    """
    results = {}
    tbody = table.find("tbody")

    for tr in tbody.find_all("tr"):
        cols = tr.find_all("td")

        ride = cols[0].get_text(strip=True)
        raw_value = cols[1].get_text(strip=True)

        # keep only numbers and dot
        cleaned = re.sub(r"[^0-9.]", "", raw_value)

        if cleaned == "":
            cleaned_value = ""
        else:
            cleaned_value = float(cleaned)

        results[ride] = cleaned_value

    return results

# ------------------------------------
# Create CSV with your exact columns
# ------------------------------------
with open(CSV_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "date", "park_id", "ride",
        "avg_queue_min", "max_queue_min", "uptime_pct"
    ])

# ------------------------------------
# Date range 2023-01-01 → today
# ------------------------------------
start_date = datetime(2023, 1, 1)
end_date = datetime.today()
current = start_date

# ------------------------------------
# Scrape loop
# ------------------------------------
while current <= end_date:

    url = build_url(current)
    print(f"Scraping {url}")

    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(res.text, "html.parser")

    # find the correct tables
    avg_table = extract_table_by_title(soup, "Average queue time")
    max_table = extract_table_by_title(soup, "Maximum queue time")
    uptime_table = extract_table_by_title(soup, "Uptime percentage")

    if not avg_table or not max_table or not uptime_table:
        print(f"⚠️ No ride data found for {current.date()}. Skipping.")
        current += timedelta(days=1)
        continue

    avg = parse_table(avg_table)
    maxw = parse_table(max_table)
    uptime = parse_table(uptime_table)

    all_rides = set(avg.keys()) | set(maxw.keys()) | set(uptime.keys())

    # write to CSV
    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        for ride in all_rides:
            writer.writerow([
                current.strftime("%Y-%m-%d"),
                PARK_ID,
                ride,
                avg.get(ride, ""),
                maxw.get(ride, ""),
                uptime.get(ride, "")
            ])

    print(f"✔ Saved {len(all_rides)} rides for {current.date()}")
    time.sleep(0.5)
    current += timedelta(days=1)

print("\n🎉 DONE! CSV saved as:", CSV_FILE)


Scraping https://queue-times.com/parks/160/calendar/2023/01/01
✔ Saved 41 rides for 2023-01-01
Scraping https://queue-times.com/parks/160/calendar/2023/01/02
✔ Saved 41 rides for 2023-01-02
Scraping https://queue-times.com/parks/160/calendar/2023/01/03
✔ Saved 41 rides for 2023-01-03
Scraping https://queue-times.com/parks/160/calendar/2023/01/04
✔ Saved 41 rides for 2023-01-04
Scraping https://queue-times.com/parks/160/calendar/2023/01/05
✔ Saved 41 rides for 2023-01-05
Scraping https://queue-times.com/parks/160/calendar/2023/01/06
✔ Saved 41 rides for 2023-01-06
Scraping https://queue-times.com/parks/160/calendar/2023/01/07
✔ Saved 41 rides for 2023-01-07
Scraping https://queue-times.com/parks/160/calendar/2023/01/08
✔ Saved 41 rides for 2023-01-08
Scraping https://queue-times.com/parks/160/calendar/2023/01/09
✔ Saved 41 rides for 2023-01-09
Scraping https://queue-times.com/parks/160/calendar/2023/01/10
✔ Saved 41 rides for 2023-01-10
Scraping https://queue-times.com/parks/160/calenda